In [1]:
SCHEDULED = {
    "Fueling": 14.57,
    "Baggage_Loading": 11.42,
    "Baggage_Unloading": 9.32,
    "Boarding": 11.17,
    "Deboarding": 2.51
}

ACTIVITIES = list(SCHEDULED.keys())


In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

rows = []

NUM_VIDEOS = 400

for _ in range(NUM_VIDEOS):
    video_length = np.random.randint(6, 15)

    # Ground Power connection happens at a random minute
    gp_start = np.random.randint(0, 3)

    for t in range(video_length):

        # Time since ground power connection
        gp_minute = max(0, t - gp_start)

        row = {"GP_Minute": gp_minute}

        for act, sched in SCHEDULED.items():

            if t < gp_start:
                progress = 0.0
            else:
                speed = np.random.uniform(0.6, 1.3)
                progress = min(gp_minute * speed / sched, 1.2)

            expected = gp_minute / sched
            deviation = progress - expected
            delay = max(0, (progress < expected) * abs(deviation) * sched)

            row[f"{act}_Progress_Ratio"] = progress
            row[f"{act}_Progress_Deviation"] = deviation
            row[f"{act}_Delay"] = delay

        rows.append(row)

df = pd.DataFrame(rows)


In [3]:
# Save the dataset
csv_file = "aircraft_turnaround_dataset.csv"
df.to_csv(csv_file, index=False)
print(f"✅ Dataset saved as {csv_file}")

# Print first 10 rows
print(df.head(10))

✅ Dataset saved as aircraft_turnaround_dataset.csv
   GP_Minute  Fueling_Progress_Ratio  Fueling_Progress_Deviation  \
0          0                0.000000                    0.000000   
1          1                0.063245                   -0.005390   
2          2                0.151736                    0.014468   
3          3                0.211700                    0.005798   
4          4                0.173690                   -0.100847   
5          5                0.297777                   -0.045393   
6          6                0.376946                   -0.034860   
7          7                0.293633                   -0.186806   
8          8                0.649684                    0.100610   
9          9                0.554460                   -0.063248   

   Fueling_Delay  Baggage_Loading_Progress_Ratio  \
0       0.000000                        0.000000   
1       0.078526                        0.072994   
2       0.000000                        0.22

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split

FEATURES = ["GP_Minute"]

for act in ACTIVITIES:
    FEATURES += [
        f"{act}_Progress_Ratio",
        f"{act}_Progress_Deviation"
    ]

TARGETS = [f"{act}_Delay" for act in ACTIVITIES]

X = df[FEATURES]
y = df[TARGETS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        random_state=42
    )
)

model.fit(X_train, y_train)


MultiOutputRegressor(estimator=RandomForestRegressor(max_depth=12,
                                                     n_estimators=300,
                                                     random_state=42))

In [5]:
import joblib

joblib.dump(model, "aircraft_delay_predictor.pkl")
print("✅ Model saved successfully")

✅ Model saved successfully
